# API Testing and Validation

In this notebook, the FastAPI inference service developed in Notebook 13 is tested and validated before deployment.

The objective is to verify that the API correctly:

- Loads the MLflow champion uplift model.
- Accepts the required model features.
- Validates incoming request data.
- Generates uplift predictions.
- Produces treatment recommendations.
- Returns responses in the expected structure.
- Handles invalid input appropriately.

The testing workflow covers both successful prediction requests and invalid inputs.

The overall workflow is:

API Request → Input Validation → Model Inference → Uplift Prediction → Treatment Recommendation → API Response

This stage helps ensure that the machine learning inference service behaves reliably before containerization and deployment.

In [1]:
import sys
import json
import numpy as np
import pandas as pd

from pathlib import Path

from fastapi.testclient import TestClient

print("Python version:")
print(sys.version)

Python version:
3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


In [2]:
PROJECT_ROOT = Path(
    r"C:\Users\ugand\customer-churn-uplift-modeling"
)

API_DIR = (
    PROJECT_ROOT / "api"
)

API_FILE = (
    API_DIR / "app.py"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nAPI file:")
print(API_FILE)

print(
    "\nAPI file exists:",
    API_FILE.exists()
)

Project root:
C:\Users\ugand\customer-churn-uplift-modeling

API file:
C:\Users\ugand\customer-churn-uplift-modeling\api\app.py

API file exists: True


In [3]:
sys.path.insert(
    0,
    str(API_DIR)
)

from app import app

print(
    "FastAPI application imported successfully."
)

C:\Users\ugand\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:151: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
C:\Users\ugand\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:186: UserWarning: Field name "schema" shadows an attribute in parent "BaseModel"; 
  warnings.warn(


FastAPI application imported successfully.


C:\Users\ugand\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:151: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
C:\Users\ugand\AppData\Local\Programs\Python\Python310\lib\site-packages\pydantic\_internal\_fields.py:151: UserWarning: Field "model_alias" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [4]:
client = TestClient(app)

print(
    "FastAPI test client created successfully."
)

FastAPI test client created successfully.


In [5]:
response = client.get(
    "/health"
)

print(
    "Status code:",
    response.status_code
)

print(
    "\nResponse:"
)

print(
    response.json()
)

Status code: 200

Response:
{'status': 'healthy', 'model': 'customer_uplift_t_learner', 'alias': 'champion'}


In [6]:
health_data = response.json()

assert response.status_code == 200

assert (
    health_data["status"]
    == "healthy"
)

assert (
    health_data["model"]
    == "customer_uplift_t_learner"
)

assert (
    health_data["alias"]
    == "champion"
)

print(
    "Health endpoint validation: PASS"
)

Health endpoint validation: PASS


In [7]:
DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "criteo-research-uplift-v2.1.csv.gz"
)

test_data = pd.read_csv(
    DATA_PATH,
    compression="gzip",
    nrows=1
)

print(
    "Test data shape:",
    test_data.shape
)

display(
    test_data.head()
)

Test data shape: (1, 16)


,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure
0,12.616365,10.059654,8.976429,4.679882,10.280525,4.115453,0.294443,4.833815,3.955396,13.190056,5.300375,-0.168679,1,0,0,0


In [8]:
FEATURE_COLUMNS = [
    f"f{i}"
    for i in range(12)
]

print(
    "API features:"
)

print(
    FEATURE_COLUMNS
)

API features:
['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11']


In [9]:
sample_row = test_data.iloc[0]

valid_payload = {
    feature: float(
        sample_row[feature]
    )
    for feature in FEATURE_COLUMNS
}

print(
    json.dumps(
        valid_payload,
        indent=4
    )
)

{
    "f0": 12.616364906986496,
    "f1": 10.059654474774549,
    "f2": 8.976428838331028,
    "f3": 4.679881620097284,
    "f4": 10.280525225748212,
    "f5": 4.115453421277861,
    "f6": 0.294442711255606,
    "f7": 4.833814577796811,
    "f8": 3.9553959684262416,
    "f9": 13.190055934673358,
    "f10": 5.300374864042156,
    "f11": -0.1686792210005612
}


In [10]:
response = client.post(
    "/predict",
    json=valid_payload
)

print(
    "Status code:",
    response.status_code
)

print(
    "\nResponse:"
)

print(
    response.json()
)

Status code: 200

Response:
{'predicted_uplift': -0.004079329291847572, 'recommendation': 'DO_NOT_TREAT', 'model_name': 'customer_uplift_t_learner', 'model_alias': 'champion'}


In [11]:
prediction_response = response.json()

assert response.status_code == 200

assert (
    "predicted_uplift"
    in prediction_response
)

assert (
    "recommendation"
    in prediction_response
)

assert (
    "model_name"
    in prediction_response
)

assert (
    "model_alias"
    in prediction_response
)

print(
    "Prediction response validation: PASS"
)

Prediction response validation: PASS


In [12]:
uplift = (
    prediction_response[
        "predicted_uplift"
    ]
)

recommendation = (
    prediction_response[
        "recommendation"
    ]
)

print(
    "Predicted uplift:",
    uplift
)

print(
    "Type:",
    type(uplift)
)

print(
    "Recommendation:",
    recommendation
)

Predicted uplift: -0.004079329291847572
Type: <class 'float'>
Recommendation: DO_NOT_TREAT


In [13]:
expected_recommendation = (
    "TREAT"
    if uplift > 0
    else "DO_NOT_TREAT"
)

assert (
    recommendation
    == expected_recommendation
)

print(
    "Treatment recommendation validation: PASS"
)

Treatment recommendation validation: PASS


In [14]:
invalid_payload = valid_payload.copy()

invalid_payload.pop("f11")

response = client.post(
    "/predict",
    json=invalid_payload
)

print(
    "Status code:",
    response.status_code
)

print(
    "\nResponse:"
)

print(
    response.json()
)

Status code: 422

Response:
{'detail': [{'type': 'missing', 'loc': ['body', 'f11'], 'msg': 'Field required', 'input': {'f0': 12.616364906986496, 'f1': 10.059654474774549, 'f2': 8.976428838331028, 'f3': 4.679881620097284, 'f4': 10.280525225748212, 'f5': 4.115453421277861, 'f6': 0.294442711255606, 'f7': 4.833814577796811, 'f8': 3.9553959684262416, 'f9': 13.190055934673358, 'f10': 5.300374864042156}, 'url': 'https://errors.pydantic.dev/2.6/v/missing'}]}


In [15]:
assert (
    response.status_code
    == 422
)

print(
    "Missing feature validation: PASS"
)

Missing feature validation: PASS


In [16]:
invalid_type_payload = valid_payload.copy()

invalid_type_payload["f0"] = (
    "not_a_number"
)

response = client.post(
    "/predict",
    json=invalid_type_payload
)

print(
    "Status code:",
    response.status_code
)

print(
    "\nResponse:"
)

print(
    response.json()
)

Status code: 422

Response:
{'detail': [{'type': 'float_parsing', 'loc': ['body', 'f0'], 'msg': 'Input should be a valid number, unable to parse string as a number', 'input': 'not_a_number', 'url': 'https://errors.pydantic.dev/2.6/v/float_parsing'}]}


In [17]:
assert (
    response.status_code
    == 422
)

print(
    "Invalid data type validation: PASS"
)

Invalid data type validation: PASS


In [18]:
test_batch = pd.read_csv(
    DATA_PATH,
    compression="gzip",
    nrows=5
)

results = []

for _, row in test_batch.iterrows():

    payload = {
        feature: float(
            row[feature]
        )
        for feature in FEATURE_COLUMNS
    }

    response = client.post(
        "/predict",
        json=payload
    )

    results.append({
        "status_code":
            response.status_code,

        "predicted_uplift":
            response.json()[
                "predicted_uplift"
            ],

        "recommendation":
            response.json()[
                "recommendation"
            ]
    })

api_results = pd.DataFrame(
    results
)

display(api_results)

,status_code,predicted_uplift,recommendation
0,200,-0.004079,DO_NOT_TREAT
1,200,-0.004079,DO_NOT_TREAT
2,200,-0.004079,DO_NOT_TREAT
3,200,-0.004079,DO_NOT_TREAT
4,200,-0.013535,DO_NOT_TREAT


In [19]:
assert (
    len(api_results)
    == 5
)

assert (
    (
        api_results[
            "status_code"
        ] == 200
    ).all()
)

assert (
    api_results[
        "predicted_uplift"
    ].notna().all()
)

print(
    "Multiple request validation: PASS"
)

Multiple request validation: PASS


In [20]:
print(
    "Prediction summary:"
)

display(
    api_results[
        "predicted_uplift"
    ].describe()
)

Prediction summary:


count    5.000000
mean    -0.005971
std      0.004229
min     -0.013535
25%     -0.004079
50%     -0.004079
75%     -0.004079
max     -0.004079
Name: predicted_uplift, dtype: float64

In [21]:
recommendation_counts = (
    api_results[
        "recommendation"
    ]
    .value_counts()
)

display(
    recommendation_counts
)

recommendation
DO_NOT_TREAT    5
Name: count, dtype: int64

In [22]:
test_results = {
    "Health endpoint": True,
    "Valid prediction request": True,
    "Prediction response schema": True,
    "Treatment recommendation": True,
    "Missing feature validation": True,
    "Invalid data type validation": True,
    "Multiple requests": True
}

print(
    "API Test Summary\n"
)

for test_name, status in (
    test_results.items()
):

    print(
        f"{test_name}: "
        f"{'PASS' if status else 'FAIL'}"
    )

API Test Summary

Health endpoint: PASS
Valid prediction request: PASS
Prediction response schema: PASS
Treatment recommendation: PASS
Missing feature validation: PASS
Invalid data type validation: PASS
Multiple requests: PASS


In [23]:
REPORTS_DIR = (
    PROJECT_ROOT / "reports"
)

REPORTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

api_test_report = pd.DataFrame(
    [
        {
            "test": test_name,
            "status": (
                "PASS"
                if status
                else "FAIL"
            )
        }
        for test_name, status
        in test_results.items()
    ]
)

API_TEST_REPORT_PATH = (
    REPORTS_DIR /
    "api_test_report.csv"
)

api_test_report.to_csv(
    API_TEST_REPORT_PATH,
    index=False
)

print(
    "API test report saved to:"
)

print(
    API_TEST_REPORT_PATH
)

API test report saved to:
C:\Users\ugand\customer-churn-uplift-modeling\reports\api_test_report.csv


In [24]:
print(
    "Final API validation\n"
)

all_tests_passed = all(
    test_results.values()
)

print(
    "All tests passed:",
    all_tests_passed
)

print(
    "Report exists:",
    API_TEST_REPORT_PATH.exists()
)

Final API validation

All tests passed: True
Report exists: True


# Conclusion

In this notebook, the FastAPI uplift inference service was systematically tested and validated.

The main outcomes were:

- Successfully imported the FastAPI application.
- Created a FastAPI test client for local endpoint testing.
- Verified the `/health` endpoint.
- Tested valid requests to the `/predict` endpoint.
- Validated the prediction response structure.
- Verified that predicted uplift values and treatment recommendations are returned correctly.
- Tested API behavior when a required feature is missing.
- Tested API behavior when an invalid data type is provided.
- Tested multiple prediction requests.
- Generated an API test report and stored it in the `reports` directory.

The validation workflow confirms that the API can accept model features, communicate with the MLflow champion model, generate uplift predictions, and return an actionable treatment recommendation.

The resulting architecture is:

Client Request
→ FastAPI Validation
→ MLflow Champion Model
→ T-Learner Inference
→ Predicted Uplift
→ Treatment Recommendation
→ Validated API Response

The API is now ready for the next stage: **containerization and deployment**, where the inference service will be packaged into a reproducible environment using Docker.